<a href="https://colab.research.google.com/github/AlessandroCaforio/Academic-Writing/blob/main/CrossWalk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd


In [11]:
comuni = pd.read_csv("/content/open_csv_20190601_comuni.tab", sep = "\t")
ANPR = pd.read_csv("/content/ANPR_archivio_comuni.csv")


In [12]:
ANPR.head()

,ID,DATAISTITUZIONE,DATACESSAZIONE,CODISTAT,CODCATASTALE,DENOMINAZIONE_IT,DENOMTRASLITTERATA,ALTRADENOMINAZIONE,ALTRADENOMTRASLITTERATA,ID_PROVINCIA,IDPROVINCIAISTAT,IDREGIONE,IDPREFETTURA,STATO,SIGLAPROVINCIA,FONTE,DATAULTIMOAGG,COD_DENOM
0,12560,1866-11-19,1924-11-13,28001,A001,ABANO,ABANO,NaN,NaN,28,28,05,NaN,C,PD,NaN,2016-06-17,28500.0
1,1,1924-11-14,9999-12-31,28001,A001,ABANO TERME,ABANO TERME,NaN,NaN,28,28,05,PD,A,PD,NaN,2016-06-17,NaN
2,3,1864-04-01,1928-12-05,1801,A003,ABBADIA ALPINA,ABBADIA ALPINA,NaN,NaN,1,1,01,NaN,C,TO,NaN,2017-11-11,NaN
3,5,1861-03-17,1992-04-15,15001,A004,ABBADIA CERRETO,ABBADIA CERRETO,NaN,NaN,15,15,03,NaN,C,MI,NaN,2016-06-17,NaN
4,4,1992-04-16,9999-12-31,98001,A004,ABBADIA CERRETO,ABBADIA CERRETO,NaN,NaN,98,98,03,LO,A,LO,NaN,2016-06-17,NaN


Nella colonna "event_to_creation_flag" abbiamo che l'evento è S o N. S sta per fusione, N sta per assorbimento.

In [13]:
M50 = pd.read_csv("/content/pre-war-covariates-complete.csv")
M21 = pd.read_csv("/content/italia_comuni_1921.csv")

def clean_name(name):
  if pd.isna(name): return ""
  return str(name).upper().replace("'", "").strip()
M50['key']= M50['comune_matchable_50'].apply(clean_name)
M21['key']= M21['nome_comune'].apply(clean_name)




# Nuovo merge basato sulle colonne normalizzate
comparison = pd.merge(
    M50[['key']],
    M21[['key']],
    on='key',
    how='outer',
    indicator=True
)

print("Risultati del confronto (dopo normalizzazione):")
print(comparison['_merge'].value_counts())

# Esempi di chi ancora non matcha
print("\nAncora solo in M50 (esempi):")
print(comparison[comparison['_merge'] == 'left_only']['key'].head())

print("\nAncora solo in M21 (esempi):")
print(comparison[comparison['_merge'] == 'right_only']['key'].head())

Risultati del confronto (dopo normalizzazione):
_merge
both          6970
right_only    2517
left_only      863
Name: count, dtype: int64

Ancora solo in M50 (esempi):
1         ABANO TERME
4     ABBADIA LARIANA
7           ABBASANTA
12            ABETONE
14              ACATE
Name: key, dtype: object

Ancora solo in M21 (esempi):
0                       
2         ABBADIA ALPINA
6     ABBADIA SOPRA ADDA
8             ABBASSANTA
10      ABBIATE GUAZZONE
Name: key, dtype: object


In [14]:
M50.head()
M50.shape

(7770, 9)

In [15]:
# Corretto: il nome della variabile non può iniziare con un numero
pop50 = M50[M50['popres_1921_tot'].isna()].copy()
pop50.head()
pop50.shape

popM = pd.merge(
    pop50['key'],
    M21['key'],
    how = 'outer',
    indicator = True
)
# Esempi di chi ancora non matcha
print(popM['_merge'].value_counts())

print("\nAncora solo in M50 (esempi):")
print(popM[popM['_merge'] == 'left_only']['key'].head())

_merge
right_only    7306
both          2177
left_only      284
Name: count, dtype: int64

Ancora solo in M50 (esempi):
9               ABETONE
11                ACATE
94       AIELLO CALABRO
95    AIELLO DEL FRIULI
98                AIETA
Name: key, dtype: object


In [16]:
ANPR['nome_pulito'] = ANPR['DENOMINAZIONE_IT'].apply(clean_name)

c_recenti = ANPR[ANPR['DATACESSAZIONE'] == '9999-12-31'][['CODISTAT', 'nome_pulito']].drop_duplicates()
c_recenti.rename(columns={'nome_pulito': 'nome_1950'}, inplace=True)

mappa_nomi = pd.merge(
    ANPR[['CODISTAT', 'nome_pulito']],
    c_recenti,
    on = 'CODISTAT'
    )


mappa_nomi = mappa_nomi[mappa_nomi['nome_pulito'] != mappa_nomi['nome_1950']]

dizionario_cambi_nome = dict(zip(mappa_nomi['nome_pulito'], mappa_nomi['nome_1950']))

def aggiorna_nome(nome_1921):
    return dizionario_cambi_nome.get(nome_1921, nome_1921)

M21['key_updated'] = M21['key'].apply(aggiorna_nome)


In [19]:
# Proviamo il merge aggiornato usando la nuova chiave 'key_updated' per il 1921
# e 'key' per il 1950
M_merged_step1 = pd.merge(
    pop50,
    M21,
    left_on='key',
    right_on='key_updated',
    how='outer',
    indicator='merge_status_1'
)

print(M_merged_step1['merge_status_1'].value_counts())

# Comuni del 1921 che ancora non trovano corrispondenza nel 1950
orfani_1921 = M_merged_step1[M_merged_step1['merge_status_1'] == 'right_only'].copy()

# Comuni del 1950 che non hanno trovato il loro passato nel 1921
orfani_1950 = M_merged_step1[M_merged_step1['merge_status_1'] == 'left_only'].copy()

merge_status_1
right_only    7309
both          2174
left_only      324
Name: count, dtype: int64


In [20]:
extinction = pd.read_csv("/content/open_csv_20190601_event_extinction.tab", sep = "\t")
print(extinction.head())

# Numero di elementi unici per colonna
print("Conteggio elementi unici:")
print(extinction.nunique())

# Visualizzazione degli elementi unici per ogni colonna
print("\nValori unici per colonna:")
COL = ["event_to_name"]

for col in COL:
    print(f"{col}: {extinction[col].unique()}")

##
ext_merged = pd.merge(
    extinction,
    comuni[['sistat_id', 'last_name']],
    on = 'sistat_id',
    how = 'inner'
    )
ext_merged['nome_soppresso_pulito'] = ext_merged['last_name'].apply(clean_name)
ext_merged['nome_incorporante_pulito'] = ext_merged['event_to_name'].apply(clean_name)

#Building a dictionary:

dizionario_soppressioni = dict(zip(
    ext_merged['nome_soppresso_pulito'],
    ext_merged['nome_incorporante_pulito']
))

   sistat_id  event_num                       event_to_name  \
0       9759       2509                            Pinerolo   
1      10260       3023                     Abbadia Lariana   
2       7865       5777                            Ghilarza   
3       6230       5144  San Valentino in Abruzzo Citeriore   
4      11522       2410                                   -   

   event_to_istat_cod    event_to_area event_to_population  \
0              1191.0  NON DOCUMENTATA     NON DOCUMENTATA   
1             13001.0  NON DOCUMENTATA     NON DOCUMENTATA   
2             92045.0  NON DOCUMENTATA     NON DOCUMENTATA   
3             68038.0  NON DOCUMENTATA     NON DOCUMENTATA   
4                 NaN  NON DOCUMENTATA     NON DOCUMENTATA   

  event_to_creation_flag  
0                      N  
1                      S  
2                      N  
3                      N  
4                      N  
Conteggio elementi unici:
sistat_id                 3277
event_num                 163

In [21]:
def applica_soppressioni(row):
    # Lavoriamo solo su quelli che non hanno ancora matchato nel 1950
    # e proviamo a vedere se sono stati assorbiti
    nome_attuale = row['key_updated']

    # Se il comune è nel nostro dizionario delle soppressioni, restituiamo il nome del comune che lo ha inglobato
    if nome_attuale in dizionario_soppressioni:
        return dizionario_soppressioni[nome_attuale]

    return nome_attuale

# Aggiorniamo la chiave per fare un "ponte" dal 1921 al 1950
M21['key_1950_target'] = M21.apply(applica_soppressioni, axis=1)

In [22]:
# Raggruppiamo i dati del 1921 per la nuova chiave di destinazione (che simula i confini del 1950)
# Assicurati di selezionare le colonne numeriche corrette da sommare (es. pop_tot, pop_M, ecc.)

colonne_da_sommare = ['pop_tot', 'pop_M', 'pop_F', 'pop_tot_alfabeti', 'pop_tot_analfabeti']

M21_aggregato = M21.groupby('key_1950_target')[colonne_da_sommare].sum().reset_index()

# Ora M21_aggregato ha una struttura "armonizzata" ai confini del 1950.
# Possiamo fare il merge definitivo con M50!

dataset_finale = pd.merge(
    M50,
    M21_aggregato,
    left_on='key',
    right_on='key_1950_target',
    how='left'
)